# Locali fine-tuning data: a look inside

Four versions of the data used to fine-tune [Locali](https://github.com/MedGm/Locali), a software-engineering assistant that runs a small code model (Qwen2.5-Coder 1.5B / 3B) on a laptop. Each version was built to fix what the previous one got wrong; the dataset pages explain why:

| Version | What it added | Main result |
|---|---|---|
| [v1](https://www.kaggle.com/datasets/elgorrimmohamed/locali-mix-v1) | repair of injected bugs, test writing, review, replay | tests 46 → 84 % of bugs caught, but code reasoning dropped |
| [v2](https://www.kaggle.com/datasets/elgorrimmohamed/locali-mix-v2) | execution traces, 5 more bug kinds, verified clean reviews | 1.5B: repair 25 → 36 %, reasoning drop gone |
| [v3](https://www.kaggle.com/datasets/elgorrimmohamed/locali-mix-v3) | 3,220 realistic bugs written by a 9B teacher, verified by tests | 3B: repair 46 → 65 % |
| [v3b](https://www.kaggle.com/datasets/elgorrimmohamed/locali-mix-v3b) | half the one-line mutations, twice the replay | 3B: repair 61 %, refactoring restored |

This notebook loads all four and shows what is in them.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

def find(version):
    for p in Path('/kaggle/input').rglob('train.jsonl'):
        if p.parent.name == f'locali-mix-{version}' or p.parent.parent.name == f'locali-mix-{version}':
            return p.parent
    raise FileNotFoundError(version)

def load(version, split):
    return [json.loads(line) for line in (find(version) / f'{split}.jsonl').read_text().splitlines() if line.strip()]

versions = ['v1', 'v2', 'v3', 'v3b']
data = {v: {s: load(v, s) for s in ('train', 'val', 'test')} for v in versions}

## Training examples per task and version

T1: repair of injected one-line bugs · T1R: realistic bugs (v3 on) · T2: test writing · T3: review · T4: execution traces · replay: general code instructions

In [ ]:
counts = pd.DataFrame({v: Counter(r['task'] for r in data[v]['train']) for v in versions}).fillna(0).astype(int)
counts.loc['total'] = counts.sum()
counts

## Answer lengths (characters), training split of v3b

In [ ]:
rows = [{'task': r['task'], 'prompt': len(r['messages'][0]['content']), 'answer': len(r['messages'][-1]['content'])}
        for r in data['v3b']['train']]
pd.DataFrame(rows).groupby('task')[['prompt', 'answer']].median()

## One example of each task (v3b, test split)

In [ ]:
seen = set()
for r in data['v3b']['test']:
    if r['task'] in seen or (r['task'] == 'T1R' and r.get('format') != 'repair'):
        continue
    seen.add(r['task'])
    print('=' * 100)
    print(f"{r['task']}   id={r['id']}   problem={r['group']}")
    for m in r['messages']:
        print(f"--- {m['role']}")
        print(m['content'][:4000] + (' [...]' if len(m['content']) > 4000 else ''))

## The realistic bugs (T1R): kinds and prompt formats

In [ ]:
t1r = [r for r in data['v3']['train'] if r['task'] == 'T1R']
print(Counter(r.get('kind') for r in t1r))
print(Counter(r.get('format') for r in t1r))

Code, data builders, training jobs and every evaluation result: [github.com/MedGm/Locali](https://github.com/MedGm/Locali).